In [ ]:

# 1. Install and import libraries
!pip install -q torch torchvision

import torch
import torchvision
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
from torchvision.transforms.functional import to_tensor
from torchvision.utils import draw_bounding_boxes

# 2. Load pretrained Faster R-CNN model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    weights=weights
)

model = model.to(device)
model.eval()

# COCO class names
categories = weights.meta["categories"]

print("Device:", device)
print("Number of classes:", len(categories))

# 3. Upload a food image
uploaded = files.upload()

image_path = next(iter(uploaded))
image = Image.open(image_path).convert("RGB")

image_tensor = to_tensor(image).to(device)

print("Image size:", image.size)

# 4. Perform object detection
with torch.no_grad():
    prediction = model([image_tensor])[0]

# Confidence threshold
threshold = 0.5

keep = prediction["scores"] >= threshold

boxes = prediction["boxes"][keep].cpu()
labels = prediction["labels"][keep].cpu()
scores = prediction["scores"][keep].cpu()

# 5. Display detected objects
print("Detected objects:")

for label, score in zip(labels, scores):
    print(f"{categories[label]}: {score:.2f}")

# 6. Draw bounding boxes
image_for_display = to_tensor(image).mul(255).byte()

box_labels = [
    f"{categories[label]}: {score:.2f}"
    for label, score in zip(labels.tolist(), scores.tolist())
]

annotated_image = draw_bounding_boxes(
    image_for_display,
    boxes=boxes,
    labels=box_labels,
    colors="red",
    width=3
)

# 7. Show final output
plt.figure(figsize=(12, 8))
plt.imshow(annotated_image.permute(1, 2, 0))
plt.axis("off")
plt.title("Food Object Detection using Faster R-CNN")
plt.show()

print("Object detection completed successfully!")
